# MLOps Lab 7: Serving a Model as a Web Service with FastAPI

## 1. Aim

To make a trained machine learning model available to other software by building a web service with FastAPI that validates incoming data and returns predictions.

## 2. Theory

### 2.1 The problem

A model that exists only inside a notebook cannot be used by anyone else. A food-delivery app, for example, needs to send the details of an order and receive a predicted delivery time. To make this possible, the model is placed behind a **web service**.

### 2.2 APIs and REST

Software communicates with other software through an **API** (Application Programming Interface). A web API has four basic parts:

- **Endpoint**: an address on the service, for example `/predict`
- **Method**: `GET` fetches information; `POST` sends data and receives a result
- **Body**: the data sent with a request, in **JSON** format, which looks like a Python dictionary
- **Status code**: the outcome of the request. `200` means success, `422` means the input was invalid, and `500` means the service failed

### 2.3 FastAPI

FastAPI is a Python framework for building web services. It **validates input automatically**, rejecting requests with missing or impossible values before they reach the model, and it **generates a documentation page** at `/docs`, where every endpoint can be tried in the browser.

Input validation matters because a model does not know when its input is unreasonable. A model trained on distances of 1 to 15 km will happily return a meaningless prediction for 500 km.

## 3. Procedure Overview

1. Train a model and save it
2. Write the web service
3. Test the service inside the notebook
4. Observe how invalid input is rejected
5. Run the service as a real server and open the documentation page
6. Complete the task: Iris Flower Classification Service

**Installation note.** If FastAPI is not yet installed, run the following command once in a terminal:

```
pip install fastapi uvicorn httpx
```

## Step 1: Train a Model and Save It

We create a synthetic food-delivery dataset of 500 orders with three features: `distance_km` (1 to 15 km), `items` (1 to 9) and `traffic_level` (1 = low, 2 = medium, 3 = high). We train a Linear Regression model and save it to `work/model.joblib`. The web service will load the model from this file.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression

# Create the dataset (fixed seed so everyone gets the same data)
np.random.seed(42)
n = 500

df = pd.DataFrame()
df["distance_km"]   = np.random.uniform(1, 15, n)
df["items"]         = np.random.randint(1, 10, n)
df["traffic_level"] = np.random.randint(1, 4, n)
df["delivery_minutes"] = (10 + 3 * df["distance_km"] + 1.5 * df["items"]
                          + 5 * df["traffic_level"] + np.random.normal(0, 3, n))

# Train the model
X = df[["distance_km", "items", "traffic_level"]]
y = df["delivery_minutes"]

model = LinearRegression()
model.fit(X, y)

# Save the trained model in the "work" folder
os.makedirs("work", exist_ok=True)
joblib.dump(model, "work/model.joblib")
print("Model saved to work/model.joblib")

Model saved to work/model.joblib


## Step 2: Write the Web Service

The service is saved as a Python file, `work/app.py`, using the command `%%writefile`. It contains:

- **`Order`**: a Pydantic class that describes a valid request. `Field(ge=..., le=...)` sets the allowed range (`ge` = greater than or equal to, `le` = less than or equal to, `gt` = greater than).
- **`GET /health`**: returns `{"status": "ok"}` to show that the service is running.
- **`POST /predict`**: receives an order and returns the predicted delivery time.

In [2]:
%%writefile work/app.py
import os
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

# Load the saved model once, when the service starts
model = joblib.load(os.path.join(os.path.dirname(__file__), "model.joblib"))

app = FastAPI(title="Delivery Time Prediction Service")


# The shape of a valid request (ranges match the training data)
class Order(BaseModel):
    distance_km:   float = Field(gt=0, le=15)
    items:         int   = Field(ge=1, le=9)
    traffic_level: int   = Field(ge=1, le=3)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(order: Order):
    # Convert the order into a one-row table, as the model expects
    row = pd.DataFrame([order.model_dump()])
    prediction = model.predict(row)[0]
    return {"predicted_minutes": round(float(prediction), 1)}

Writing work/app.py


## Step 3: Test the Service Inside the Notebook

FastAPI's **TestClient** sends requests directly to the application, so no server is needed. We call both endpoints and print the status code and reply.

In [3]:
import sys
import importlib
from fastapi.testclient import TestClient

# Load app.py from the "work" folder (reload picks up any edits to the file)
sys.path.insert(0, "work")
import app
importlib.reload(app)

client = TestClient(app.app)

# Check that the service is running
response = client.get("/health")
print("Health  :", response.status_code, response.json())

# Ask for a prediction
order = {"distance_km": 8, "items": 3, "traffic_level": 2}
response = client.post("/predict", json=order)
print("Predict :", response.status_code, response.json())

Health  : 200 {'status': 'ok'}
Predict : 200 {'predicted_minutes': 48.4}


## Step 4: Observe How Invalid Input Is Rejected

We send an order with an impossible traffic level (9). The service replies with status code **422** and explains the problem. The model is never called.

In [4]:
bad_order = {"distance_km": 8, "items": 3, "traffic_level": 9}

response = client.post("/predict", json=bad_order)

print("Status code:", response.status_code)
print("Problem    :", response.json()["detail"][0]["msg"])

Status code: 422
Problem    : Input should be less than or equal to 3


## Step 5: Run the Service as a Real Server

We now start the service as a real web server using **uvicorn**. **Run this cell only once**; running it again fails because port 8000 is already in use.

After about 5 seconds, open **http://127.0.0.1:8000/docs** in a web browser. Open **POST /predict**, click **Try it out**, enter an order, and click **Execute**. The prediction appears in the **Response body** panel.

**Alternative.** The server can also be started from a terminal opened in the `work` folder:

```
python -m uvicorn app:app --port 8000
```

> **Windows users:** use **Anaconda Prompt**, move to the `work` folder using `cd`, and run the command above. If a firewall prompt appears, click **Allow**.

In [5]:
import subprocess

# Start the web server in the background, inside the "work" folder
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--port", "8000"],
    cwd="work",
)

print("Server is starting. Open http://127.0.0.1:8000/docs in your browser.")

Server is starting. Open http://127.0.0.1:8000/docs in your browser.


To stop the server at the end of the session, remove the `#` and run the cell below.

In [6]:
# server.terminate()

# Task: Iris Flower Classification Service

Use the **Iris dataset**, which is built into scikit-learn (load it with `from sklearn.datasets import load_iris`, so no download is needed), to build a web service that predicts the species of an iris flower from four measurements in centimetres: sepal length, sepal width, petal length and petal width. Train a `LogisticRegression(max_iter=1000)` model, save it in a folder named `task`, and write a FastAPI service in `task/iris_app.py` with two endpoints: `GET /health`, which returns `{"status": "ok"}`, and `POST /predict`, which accepts the four measurements and returns the predicted species **name** (for example `"setosa"`). Use Pydantic `Field` constraints so that every measurement must be greater than 0 and at most 10 cm. Using the TestClient, show one successful prediction and show that a request with a negative measurement is rejected with status code 422. Finally, run the service with uvicorn and take a screenshot of the `/docs` page showing a successful prediction in the **Response body** panel.

## Step 6: Task - Iris Flower Classification Service

In [7]:
import os
import joblib
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

iris = load_iris()
Xi = iris.data
yi = iris.target

iris_model = LogisticRegression(max_iter=1000)
iris_model.fit(Xi, yi)

os.makedirs("task", exist_ok=True)
joblib.dump(iris_model, "task/iris_model.joblib")
joblib.dump(list(iris.target_names), "task/iris_target_names.joblib")

print("Model saved to task/iris_model.joblib")
print("Classes:", list(iris.target_names))

Model saved to task/iris_model.joblib
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


### Write the Iris FastAPI service (`task/iris_app.py`)

In [8]:
%%writefile task/iris_app.py
import os
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

HERE = os.path.dirname(__file__)
model = joblib.load(os.path.join(HERE, "iris_model.joblib"))
target_names = joblib.load(os.path.join(HERE, "iris_target_names.joblib"))

app = FastAPI(title="Iris Flower Classification Service")


class IrisMeasurements(BaseModel):
    sepal_length: float = Field(gt=0, le=10)
    sepal_width:  float = Field(gt=0, le=10)
    petal_length: float = Field(gt=0, le=10)
    petal_width:  float = Field(gt=0, le=10)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(sample: IrisMeasurements):
    row = pd.DataFrame([[
        sample.sepal_length,
        sample.sepal_width,
        sample.petal_length,
        sample.petal_width,
    ]], columns=["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"])

    predicted_class = model.predict(row)[0]
    species = target_names[predicted_class]
    return {"species": species}

Writing task/iris_app.py


### Test the Iris service with TestClient

In [9]:
import sys
import importlib
from fastapi.testclient import TestClient

sys.path.insert(0, "task")
import iris_app
importlib.reload(iris_app)

iris_client = TestClient(iris_app.app)

# Health check
response = iris_client.get("/health")
print("Health  :", response.status_code, response.json())

# A valid sample (typical setosa measurements)
sample = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2,
}
response = iris_client.post("/predict", json=sample)
print("Predict :", response.status_code, response.json())

Health  : 200 {'status': 'ok'}
Predict : 200 {'species': 'setosa'}


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


### Show that an invalid (negative) measurement is rejected with 422

In [10]:
bad_sample = {
    "sepal_length": -1.0,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2,
}

response = iris_client.post("/predict", json=bad_sample)

print("Status code:", response.status_code)
print("Problem    :", response.json()["detail"][0]["msg"])

Status code: 422
Problem    : Input should be greater than 0


### Run the Iris service as a real server

**Run this cell only once** — running it again fails because port 8001 is already in use.

After a few seconds, open **http://127.0.0.1:8001/docs**, open **POST /predict**, click **Try it out**, enter a sample, and click **Execute**.

In [11]:
import subprocess

iris_server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "iris_app:app", "--port", "8001"],
    cwd="task",
)

print("Iris server is starting. Open http://127.0.0.1:8001/docs in your browser.")

Iris server is starting. Open http://127.0.0.1:8001/docs in your browser.


To stop the server at the end of the session, remove the `#` and run the cell below.

In [12]:
#iris_server.terminate()